# ANALYSE-06 — Sensitivity 2026: our lake facing the openai/math families (native companion)

**ANALYSE** sub-series (#17545 grading) — Fold 2 of the **Origami** digestion (#19898): the Boolean-combinatorics results published by `openai/math` in 2026, read from **our** `Sensitivity` lake — the port of Huang's theorem (2019).

This notebook is a **native** companion: Lean 4 kernel under WSL, every `#check` queries the actually-compiled lake, every `#eval` really computes. Numerical computation is done **natively in Lean** — no Python fallback; the file-name suffix follows the series naming convention. This is the English twin of the French notebook `ANALYSE-06-Sensibilite-2026-Lean.ipynb`; it renders the **same** computation in English: the code cells are byte-identical to the French side (the repository's twin-parity invariant), while the lake's English siblings (`*_en`, i18n convention #4980) carry the English docstrings.

**Prerequisites**: the main series tutorial (notebooks 1 to 6). Reading [Lean-12b — Huang's Sensitivity Theorem](../Lean-12b-Lean-Sensitivity-Theorem.ipynb) makes section 1 more comfortable, but is not required: we recall what we need.

**Plan**: our own foundation (Huang) — the four 2026 families quoted with their status — our own sensitivity/block-sensitivity separations (module `Sensitivity.Separations`) checked and computed — three exercises.

## Navigation

| Previous notebook | French version | Next notebook |
|---|---|---|
| [ANALYSE-04 — Three PFR primitives](ANALYSE-04-PFR-Primitives-Python.ipynb) | [ANALYSE-06 — French version](ANALYSE-06-Sensibilite-2026-Lean.ipynb) | [Lean-12b — Huang's Sensitivity Theorem](../Lean-12b-Lean-Sensitivity-Theorem.ipynb) |

## 0. Epistemic status — the rule of this notebook

The Origami digestion requires a **declared status** for every statement. Three statuses exist: **ours** (proven in our lake), **formalized** (proven by the openai/math authors, Lean skeleton published), **not formalized** (no formal proof). The rule: never "OpenAI proved X, therefore X" — this notebook distinguishes, on every line, what **we prove and execute** from what **we quote**.

| Object | Status | Where to verify |
|---|---|---|
| Huang's theorem (2019) | **ours** | `#check` + `#print axioms`, section 1 |
| Families 127, 132, 186, 192 (openai/math, 2026) | **formalized** (by the authors) | quoted in section 2 — skeletons `lean/docs/NNN.md` of the openai/math repository; their proofs are NOT executed here |
| `sensitivity_le_blockSensitivity`, `orOfAnds_strict_separation`, `or3_blockSensitivity` | **ours** | `#check` + `#print axioms` + `#eval`, section 3 |

What this notebook does **not** do: re-prove the 2026 results. Their scale (hundreds of pages per manuscript) puts re-proof out of reach of this repository — that is precisely why the status "formalized" exists, and why the published skeletons remain the reference.

## 1. The foundation: Huang 2019, proven on our side

The starting point of the `Sensitivity` lake is Huang's theorem (2019) and its resolution of the sensitivity conjecture: in the hypercube $Q_{m+1}$, if you color more than half of the vertices ($|H| \ge 2^m + 1$), then one of the colored vertices has at least $\sqrt{m+1}$ colored neighbors. The central consequence for us: the block sensitivity of a Boolean function is bounded by the **square** of its sensitivity, $bs(f) \le s(f)^2$.

Let us import the lake. Its globs cover the **submodules** (not the root), so we import explicitly the modules this notebook uses — the root aggregator `Sensitivity.lean` is not a compiled module of the lake. Here, English twin, we exercise the `_en` side:

In [1]:
import Sensitivity.Hypercube
import Sensitivity.MainTheorem
import Sensitivity.Fourier
import Sensitivity.Separations

open Sensitivity

import Sensitivity.Hypercube
import Sensitivity.MainTheorem
import Sensitivity.Fourier
import Sensitivity.Separations

open Sensitivity
--% env 0

Raw input:
{"cmd": "import Sensitivity.Hypercube\nimport Sensitivity.MainTheorem\nimport Sensitivity.Fourier\nimport Sensitivity.Separations\n\nopen Sensitivity"}
Raw output:
{"env": 0}

In [2]:
#check Sensitivity.huang_degree_theorem
#check Sensitivity.exists_eigenvalue

#check Sensitivity.huang_degree_theorem
──────▶  Sensitivity.huang_degree_theorem {m : ℕ} (H : Set (Q m.succ)) (hH : H.toFinset.card ≥ 2 ^ m + 1) :
  ∃ q ∈ H, √(↑m + 1) ≤ ↑(H ∩ q.adjacent).toFinset.card
#check Sensitivity.exists_eigenvalue
──────▶  Sensitivity.exists_eigenvalue {m : ℕ} (H : Set (Q m.succ)) (hH : H.toFinset.card ≥ 2 ^ m + 1) :
  ∃ y ∈ Submodule.span ℝ (e '' H) ⊓ (g m).range, y ≠ 0
--% env 1

Raw input:
{"cmd": "#check Sensitivity.huang_degree_theorem\n#check Sensitivity.exists_eigenvalue", "env": 0}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 1, "column": 0},
   "endPos": {"line": 1, "column": 6},
   "data":
   "Sensitivity.huang_degree_theorem {m : ℕ} (H : Set (Q m.succ)) (hH : H.toFinset.card ≥ 2 ^ m + 1) :\n  ∃ q ∈ H, √(↑m + 1) ≤ ↑(H ∩ q.adjacent).toFinset.card"},
  {"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "Sensitivity.exists_eigenvalue {m : ℕ} (H : Set (Q m.succ)) (hH : H.toFinset.card ≥ 2 ^ m + 1) :\n  ∃ y ∈ Submodule.span ℝ (e '' H) ⊓ (g m).range, y ≠ 0"}],
 "env": 1}

### Reading the result

`huang_degree_theorem` is the carrier statement: the spectral bound $d^2$ on the degree of the induced subgraph. `exists_eigenvalue` is the pivotal spectral lemma (existence of an eigenvalue of modulus $\sqrt{d}$ for Huang's operator $f_n$ on the free space $V_n$). The full arc — space $V_n$, operator $f_n$, nilpotence, main theorem — is walked through in [Lean-12b](../Lean-12b-Lean-Sensitivity-Theorem.ipynb); here we take the theorem as a foundation and verify its logical footprint:

In [3]:
#print axioms Sensitivity.huang_degree_theorem

#print axioms Sensitivity.huang_degree_theorem
──────▶  'Sensitivity.huang_degree_theorem' depends on axioms: [propext, Classical.choice, Quot.sound]
--% env 2

Raw input:
{"cmd": "#print axioms Sensitivity.huang_degree_theorem", "env": 1}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 1, "column": 0},
   "endPos": {"line": 1, "column": 6},
   "data":
   "'Sensitivity.huang_degree_theorem' depends on axioms: [propext, Classical.choice, Quot.sound]"}],
 "env": 2}

### No `sorry`, minimal footprint

`[propext, Classical.choice, Quot.sound]` — Mathlib's three standard axioms, no `sorryAx`, no `native_decide`. The carrier theorem is **ours** in the full sense: proven in the lake, verifiable by this command at any time.

## 2. What moved in 2026 — four families, quoted with their status

The `openai/math` repository publishes 242 result families with Lean skeletons (`lean/docs/NNN.md`). Our resonance map (Origami distillation, `30-resonance.md`) identifies four whose objects are **literally ours** — sensitivity, influences, Fourier coefficients, degree. Status of all four: **formalized** — by the authors, not by us. We quote their statements (taken from their abstracts); their proofs are not executed here.

| Family | Statement (faithful abstract) | Link to our lake |
|---|---|---|
| **127** | The **average** sensitivity of a degree-$\le d$ polynomial threshold function on the uniform cube is $\le 8d\sqrt{n}$, for $1 \le d \le n$ — the asymptotic Gotsman–Linial conjecture | The "smooth" analogue of our local sensitivity: same object, averaged over the cube |
| **132** | Constructs total Boolean functions with $bs(f) \ge s(f)^\alpha$ for a fixed $\alpha > 2$, disproving the quadratic strengthening of the Sensitivity Conjecture | The natural next question on **our** objects $s$ and $bs$ — that is section 2.1 |
| **186** | Proves the Friedgut–Kalai threshold-width conjectures: every nontrivial increasing relabeling-invariant property crosses from $\varepsilon$ to $1-\varepsilon$ within width $O((\log n)^{-2})$ (graphs), $O_r((\log n)^{-r/(r-1)})$ ($r$-uniform hypergraphs) | The bridge to influences — the $\widehat f$ / influence vocabulary of section 4 |
| **192** | Disproves the square-root bound relating linear Fourier coefficients to degree: for every $C > 0$ there is a sign-valued $f$ with $\sum_i \widehat f(\{i\}) > C\sqrt{\deg f}$ | Same coefficients $\widehat f$ as our `Fourier` module (section 4) |

Two of these families are **disproofs** (132, 192): the 2026 mood is not only to prove bounds, but to show that **natural** bounds are false. That is exactly the gesture we reproduce at our scale in section 3: exhibiting a concrete counterexample to the equality $s = bs$.

### 2.1 Family 132 — the natural next question after Huang

Huang gives $bs(f) \le s(f)^2$ (for the global variants of $s$ and $bs$). The open question becomes: **is the exponent 2 the right one?** Can it be strengthened? Family 132 answers with a counterexample: it constructs functions where block sensitivity exceeds **any strengthened bound** — in the precise sense of its definitions, which we do not transcribe here (the exact statement lives in its manuscript and its skeleton `lean/docs/132.md`).

What **we** prove, and which is the pedagogical foundation of this question: separability is **real** — there exist functions and points where $s(f, x) < bs(f, x)$ strictly. That is the subject of section 3:

In [4]:
#check Sensitivity.sensitivity_le_blockSensitivity
#check Sensitivity.orOfAnds_strict_separation

#check Sensitivity.sensitivity_le_blockSensitivity
──────▶  Sensitivity.sensitivity_le_blockSensitivity {n : ℕ} (F : Q n → Bool) (x : Q n) : sensitivity F x ≤ blockSensitivity F x
#check Sensitivity.orOfAnds_strict_separation
──────▶  Sensitivity.orOfAnds_strict_separation :
  (sensitivity orOfAnds fun x => false) < blockSensitivity orOfAnds fun x => false
--% env 3

Raw input:
{"cmd": "#check Sensitivity.sensitivity_le_blockSensitivity\n#check Sensitivity.orOfAnds_strict_separation", "env": 2}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 1, "column": 0},
   "endPos": {"line": 1, "column": 6},
   "data":
   "Sensitivity.sensitivity_le_blockSensitivity {n : ℕ} (F : Q n → Bool) (x : Q n) : sensitivity F x ≤ blockSensitivity F x"},
  {"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "Sensitivity.orOfAnds_strict_separation :\n  (sensitivity orOfAnds fun x => false) < blockSensitivity orOfAnds fun x => false"}],
 "env": 3}

### Reading — what we proved

- `sensitivity_le_blockSensitivity`: flipping one coordinate is flipping a singleton block — so $s(f, x) \le bs(f, x)$ **always**. This is the floor bound: the two quantities cannot cross.
- `orOfAnds_strict_separation`: on an explicit example in dimension 4, the inequality is **strict** — $s < bs$ at a concrete point. The gap between $s$ and $bs$ is not a theoretical curiosity: it is a phenomenon computable end to end, and we are going to compute it.

## 3. Our definitions — the module `Sensitivity.Separations`

The module added by Fold 2 defines the two quantities and the separating example. All of them live in the lake (namespace `Sensitivity`):

In [5]:
#check Sensitivity.sensitivity
#check Sensitivity.blockSensitivity
#check Sensitivity.flipBlock
#check Sensitivity.orOfAnds

#check Sensitivity.sensitivity
──────▶  Sensitivity.sensitivity {n : ℕ} (F : Q n → Bool) (x : Q n) : ℕ
#check Sensitivity.blockSensitivity
──────▶  Sensitivity.blockSensitivity {n : ℕ} (F : Q n → Bool) (x : Q n) : ℕ
#check Sensitivity.flipBlock
──────▶  Sensitivity.flipBlock {n : ℕ} (x : Q n) (S : Finset (Fin n)) : Q n
#check Sensitivity.orOfAnds
──────▶  Sensitivity.orOfAnds (x : Q 4) : Bool
--% env 4

Raw input:
{"cmd": "#check Sensitivity.sensitivity\n#check Sensitivity.blockSensitivity\n#check Sensitivity.flipBlock\n#check Sensitivity.orOfAnds", "env": 3}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 1, "column": 0},
   "endPos": {"line": 1, "column": 6},
   "data": "Sensitivity.sensitivity {n : ℕ} (F : Q n → Bool) (x : Q n) : ℕ"},
  {"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "Sensitivity.blockSensitivity {n : ℕ} (F : Q n → Bool) (x : Q n) : ℕ"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data":
   "Sensitivity.flipBlock {n : ℕ} (x : Q n) (S : Finset (Fin n)) : Q n"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 6},
   "data": "Sensitivity.orOfAnds (x : Q 4) : Bool"}],
 "env": 4}

### Reading the definitions

On the cube $Q_n = (\mathbb{Z}/2)^n$ (type `Q n = Fin n → Bool` in the lake):

- **`sensitivity F x`** — local sensitivity: the number of coordinates $i$ whose **single** flip changes the value, $s(F, x) = \#\{i : F(x \oplus e_i) \ne F(x)\}$;
- **`flipBlock x S`** — flip a **block** $S$ of coordinates at once: $(x \oplus \chi_S)$;
- **`blockSensitivity F x`** — local block sensitivity, **standard Nisan–Szegedy definition**: the **maximum number of pairwise disjoint sensitive blocks** — the largest cardinal of a family of nonempty blocks, each sensitive ($F(x \oplus \chi_S) \ne F(x)$), whose two distinct members never overlap. This is **not** the raw count of sensitive blocks: at the zero vertex of `or3` (OR on three coordinates), all seven nonempty blocks are sensitive and the standard value is **3** — the three singletons (`or3_blockSensitivity`, measured in 3.1);
- **`orOfAnds`** — the dimension-4 separating example: $(x_0 \wedge x_1) \vee (x_2 \wedge x_3)$.

A singleton is a block: $s \le bs$. The question is whether the inequality can be **strict** — and `orOfAnds` answers at the zero vertex $x = 0000$: no single coordinate turns $F$ on ($s = 0$), but the block $\{0, 1\}$ does ($bs \ge 1$).


### 3.1 Numerical experiment — native `#eval`

The computation runs **in Lean**, on the lake's **very definition**: each `#eval blockSensitivity ...` below calls `Sensitivity.blockSensitivity` — the maximum over disjoint families — not a counting reimplementation. One honest subtlety, visible in the code: `Finset.toList` is noncomputable in Mathlib; the **profiles** of $s$ and $bs$ over the cube are therefore read as **histograms** — for each value $k$, how many vertices have $s(x) = k$ (resp. $bs(x) = k$).

Six measurements: the local sensitivity of `orOfAnds` at the zero vertex; the histogram of $s$ over the 16 vertices of $Q_4$; the block sensitivity at the zero vertex; the histogram of $bs$; then, on `or3` at the zero vertex, the standard value of $bs$ **against** the raw count of sensitive blocks — the witness that distinguishes the two readings of the definition:


In [6]:
#eval sensitivity orOfAnds (fun _ => false)

#eval ((List.range 3).map fun k =>
  ((Finset.univ : Finset (Q 4)).filter (fun x => sensitivity orOfAnds x = k)).card)

#eval blockSensitivity orOfAnds (fun _ => false)

#eval ((List.range 4).map fun k =>
  ((Finset.univ : Finset (Q 4)).filter (fun x => blockSensitivity orOfAnds x = k)).card)

#eval blockSensitivity or3 (fun _ => false)

#eval ((Finset.univ : Finset (Finset (Fin 3))).filter
    (fun S : Finset (Fin 3) => 0 < S.card ∧
      or3 (flipBlock (fun _ => false) S) ≠ or3 (fun _ => false))).card


#eval sensitivity orOfAnds (fun _ => false)
─────▶  0

#eval ((List.range 3).map fun k =>
─────▶  [2, 4, 10]
  ((Finset.univ : Finset (Q 4)).filter (fun x => sensitivity orOfAnds x = k)).card)

#eval blockSensitivity orOfAnds (fun _ => false)
─────▶  2

#eval ((List.range 4).map fun k =>
─────▶  [0, 0, 16, 0]
  ((Finset.univ : Finset (Q 4)).filter (fun x => blockSensitivity orOfAnds x = k)).card)

#eval blockSensitivity or3 (fun _ => false)
─────▶  3

#eval ((Finset.univ : Finset (Finset (Fin 3))).filter
─────▶  7
    (fun S : Finset (Fin 3) => 0 < S.card ∧
      or3 (flipBlock (fun _ => false) S) ≠ or3 (fun _ => false))).card

--% env 5

Raw input:
{"cmd": "#eval sensitivity orOfAnds (fun _ => false)\n\n#eval ((List.range 3).map fun k =>\n  ((Finset.univ : Finset (Q 4)).filter (fun x => sensitivity orOfAnds x = k)).card)\n\n#eval blockSensitivity orOfAnds (fun _ => false)\n\n#eval ((List.range 4).map fun k =>\n  ((Finset.univ : Finset (Q 4)).filter (fun x => blockSensitivity orOfAnds x = k)).card)\n\n#eval blockSensitivity or3 (fun _ => false)\n\n#eval ((Finset.univ : Finset (Finset (Fin 3))).filter\n    (fun S : Finset (Fin 3) => 0 < S.card \u2227\n      or3 (flipBlock (fun _ => false) S) \u2260 or3 (fun _ => false))).card\n", "env": 4}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 1, "column": 0},
   "endPos": {"line": 1, "column": 5},
   "data": "0"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 5},
   "data": "[2, 4, 10]"},
  {"severity": "info",
   "pos": {"line": 6, "column": 0},
   "endPos": {"line": 6, "column": 5},
   "data": "2"},
  {"severity": "info",
   "pos": {"line": 8, "column": 0},
   "endPos": {"line": 8, "column": 5},
   "data": "[0, 0, 16, 0]"},
  {"severity": "info",
   "pos": {"line": 11, "column": 0},
   "endPos": {"line": 11, "column": 5},
   "data": "3"},
  {"severity": "info",
   "pos": {"line": 13, "column": 0},
   "endPos": {"line": 13, "column": 5},
   "data": "7"}],
 "env": 5}

### Reading the result

- At the zero vertex $0000$: $s = 0$ and $bs = 2$. The blocks that turn the function on contain the pair $\{0,1\}$ or the pair $\{2,3\}$ — there are many (7 in total), but **at most two can be disjoint**: $\{0,1\}$ and $\{2,3\}$. The strict separation of `orOfAnds_strict_separation`, computed on the standard definition.
- Histogram of $s$ (for $k = 0, 1, 2$): `[2, 4, 10]` — $s = 0$ at exactly two vertices, the zero vertex $0000$ **and** the all-ones vertex $1111$ (flipping one coordinate preserves the value there: the other pair stays saturated); $s$ peaks at **2** on ten vertices. The sensitivity of `orOfAnds` is small everywhere.
- Histogram of $bs$ (for $k = 0..3$): `[0, 0, 16, 0]` — $bs = 2$ on **all** vertices. Everywhere, the example offers exactly two disjoint “steps”: either the two pairs to saturate in blocks (where $F = 0$), or the two directions to break the saturated pairs (where $F = 1$). The inequality $s \le bs$ holds everywhere, with **equality** on the ten vertices where $s = 2$: on this example, it is tight.
- The `or3` witness at the zero vertex: $bs = 3$ while the raw count of sensitive blocks is **7**. The standard definition reads the maximum of disjoint blocks (the three singletons), not the enumeration — exactly the difference between the two readings, executed.

This is **the same experiment** a reader would run in Python by brute force (enumerating $2^4$ vertices and $2^4$ blocks), but executed by the Lean compiler on the lake's **own** definitions — a single source of truth, not a reimplementation.


In [7]:
#check Sensitivity.orOfAnds_sensitivity_zero
#check Sensitivity.orOfAnds_blockSensitivity_pos
#check Sensitivity.orOfAnds_strict_separation
#check Sensitivity.or3_blockSensitivity
#check Sensitivity.or3_sensitive_blocks_card

#print axioms Sensitivity.orOfAnds_strict_separation


#check Sensitivity.orOfAnds_sensitivity_zero
──────▶  Sensitivity.orOfAnds_sensitivity_zero : (sensitivity orOfAnds fun x => false) = 0
#check Sensitivity.orOfAnds_blockSensitivity_pos
──────▶  Sensitivity.orOfAnds_blockSensitivity_pos : 0 < blockSensitivity orOfAnds fun x => false
#check Sensitivity.orOfAnds_strict_separation
──────▶  Sensitivity.orOfAnds_strict_separation :
  (sensitivity orOfAnds fun x => false) < blockSensitivity orOfAnds fun x => false
#check Sensitivity.or3_blockSensitivity
──────▶  Sensitivity.or3_blockSensitivity : (blockSensitivity or3 fun x => false) = 3
#check Sensitivity.or3_sensitive_blocks_card
──────▶  Sensitivity.or3_sensitive_blocks_card :
  {S | S.Nonempty ∧ or3 (flipBlock (fun x => false) S) ≠ or3 fun x => false}.card = 7

#print axioms Sensitivity.orOfAnds_strict_separation
──────▶  'Sensitivity.orOfAnds_strict_separation' depends on axioms: [propext, Classical.choice, Quot.sound]

--% env 6

Raw input:
{"cmd": "#check Sensitivity.orOfAnds_sensitivity_zero\n#check Sensitivity.orOfAnds_blockSensitivity_pos\n#check Sensitivity.orOfAnds_strict_separation\n#check Sensitivity.or3_blockSensitivity\n#check Sensitivity.or3_sensitive_blocks_card\n\n#print axioms Sensitivity.orOfAnds_strict_separation\n", "env": 5}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 1, "column": 0},
   "endPos": {"line": 1, "column": 6},
   "data":
   "Sensitivity.orOfAnds_sensitivity_zero : (sensitivity orOfAnds fun x => false) = 0"},
  {"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "Sensitivity.orOfAnds_blockSensitivity_pos : 0 < blockSensitivity orOfAnds fun x => false"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data":
   "Sensitivity.orOfAnds_strict_separation :\n  (sensitivity orOfAnds fun x => false) < blockSensitivity orOfAnds fun x => false"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 6},
   "data":
   "Sensitivity.or3_blockSensitivity : (blockSensitivity or3 fun x => false) = 3"},
  {"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 6},
   "data":
   "Sensitivity.or3_sensitive_blocks_card :\n  {S | S.Nonempty ∧ or3 (flipBlock (fun x => false) S) ≠ or3 fun x => false}.card = 7"},
  {"severity": "info",
   "pos": {"line": 7, "column": 0},
   "endPos": {"line": 7, "column": 6},
   "data":
   "'Sensitivity.orOfAnds_strict_separation' depends on axioms: [propext, Classical.choice, Quot.sound]"}],
 "env": 6}

### The strict separation, formally

The three theorems say exactly what the computation showed: $s(\texttt{orOfAnds}, 0000) = 0$, $0 < bs(\texttt{orOfAnds}, 0000)$, hence $s < bs$ on this example. The `or3` witness theorems — `or3_blockSensitivity : bs = 3` and `or3_sensitive_blocks_card : raw count = 7` — state the difference between the two readings. The axiom footprint of `orOfAnds_strict_separation` is **identical to that of Huang's theorem** (`[propext, Classical.choice, Quot.sound]`): no new logical debt, the `Separations` module joins the lake without degrading its guarantee.


## 4. Families 127, 186, 192 — the Fourier vocabulary

The three other families speak a neighboring dialect: **average** sensitivity (127), influences and thresholds (186), **linear** Fourier coefficients (192). Our lake carries exactly that vocabulary in its `Fourier` module — these are the objects those families bound or disprove:

In [8]:
#check Sensitivity.χ
#check Sensitivity.fourierCoeff
#check Sensitivity.influence

#check Sensitivity.χ
──────▶  Sensitivity.χ {n : ℕ} (S : Finset (Fin n)) (x : Q n) : ℤ
#check Sensitivity.fourierCoeff
──────▶  Sensitivity.fourierCoeff {n : ℕ} (f : Q n → ℤ) (S : Finset (Fin n)) : ℤ
#check Sensitivity.influence
──────▶  Sensitivity.influence {n : ℕ} (F : Q n → Bool) (i : Fin n) : ℤ
--% env 7

Raw input:
{"cmd": "#check Sensitivity.\u03c7\n#check Sensitivity.fourierCoeff\n#check Sensitivity.influence", "env": 6}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 1, "column": 0},
   "endPos": {"line": 1, "column": 6},
   "data": "Sensitivity.χ {n : ℕ} (S : Finset (Fin n)) (x : Q n) : ℤ"},
  {"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "Sensitivity.fourierCoeff {n : ℕ} (f : Q n → ℤ) (S : Finset (Fin n)) : ℤ"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data": "Sensitivity.influence {n : ℕ} (F : Q n → Bool) (i : Fin n) : ℤ"}],
 "env": 7}

### Reading — one vocabulary, distinct statuses

- **$\chi$**: the characters of $Q_n$, the Fourier basis of the cube;
- **`fourierCoeff`**: the coefficient $\widehat F(\xi)$ on character $\xi$ — the quantity family **192** claims can exceed $C\sqrt{\deg f}$ for every $C$;
- **`influence`**: the influence of a coordinate, $\Pr_{x}[F(x \oplus e_i) \ne F(x)]$ — the brick family **127** bounds in sum (average sensitivity $\le 8d\sqrt n$ for polynomial threshold functions) and from which family **186** derives its threshold widths.

Status of the three families: **formalized** (by the authors). We execute **their objects** in **our** lake — the definitions above are ours and they run; their theorems remain quotations referring to their skeletons.

## 5. Exercises

### Exercise 1 — AND is locally maximally sensitive

Define `andOf3 : Q 3 → Bool` (the AND of three bits), then show **by `#eval`** (same patterns as section 3.1) that its local sensitivity at vertex $111$ is $3$ — every bit is critical — while at the zero vertex it is $0$. At which other vertex (vertices?) is the sensitivity nonzero? Explain why AND "concentrates" all its sensitivity on the all-true vertex while `orOfAnds` spreads it out.

### Exercise 2 — Equality and separation on $Q_2$

On $Q_2$, find: (a) a function where $s(x) = bs(x)$ at **every** point; (b) a nonconstant function where $s(x) < bs(x)$ at **every** point. Verify each case by `#eval` with the lake's `blockSensitivity` definition (the maximum of disjoint blocks). Does the trivial case (a) (a constant function) count? Show that for **every** function and **every** point, $s(x) \le bs(x)$ — recover the singleton argument of `sensitivity_le_blockSensitivity`.

### Exercise 3 — How far can the separation go? (on paper)

Generalize `orOfAnds`: $F_k$ = the OR of $k$ AND-blocks of $k$ bits each, in dimension $k^2$, at the zero vertex. Show on paper that $s(F_k, 0) = 0$ and $bs(F_k, 0) \ge k$: the separation $bs/s$ (in the sense of the ratio of global maxima) can become arbitrarily large. This computation sketches **why** strengthening the $s \leftrightarrow bs$ link is a genuine question — the one family 132 settles at the 2026 scale with its superquadratic counterexample. Still on paper: why does the "singleton" argument only ever give $s \le bs$, and never better?

## Conclusion

This notebook held the three registers of the Origami digestion:

1. **Our foundation** — Huang's theorem, proven in our lake, `#check` and `#print axioms` in support: status **ours**, footprint `[propext, Classical.choice, Quot.sound]`.
2. **The 2026 families quoted** — 127, 132, 186, 192: statements faithfully taken from their abstracts, status **formalized** (by the authors), skeletons `lean/docs/NNN.md` of the openai/math repository as reference — never executed nor re-proven here.
3. **Our own contribution** — the module `Sensitivity.Separations`: definitions $s$, $bs$, `flipBlock`, the floor inequality $s \le bs$ and the explicit strict separation in dimension 4, proven **and** computed (`#eval`) on the lake's own definitions.

**SOTA verdicts** (repository rule): **SOTA-OK** for this notebook — real Lean 4 + Mathlib, real compiled lake, real native `#eval`, no substitution. **INTRINSIC** for re-proving the 2026 families: out of scale for this repository — the existence of the published skeletons is precisely what makes quotation the honest gesture.

**References**: lake [`sensitivity_lean`](../sensitivity_lean/) · issues #19898 (Epic Origami) and #19908 (Fold 2) · Huang, *Induced subgraphs of hypercubes and a proof of the sensitivity conjecture*, Annals of Mathematics 190 (2019) · Gotsman & Linial, *On the influence of variables on Boolean functions* (1991) · Friedgut & Kalai (1996) · manuscripts 127/132/186/192 of the openai/math repository (2026) and their skeletons `lean/docs/NNN.md`.